# 11 — Embeddings and a tiny text classifier

**Level:** Intermediate · **Study time:** 60–90 minutes (including experiments)

**Prerequisites:** Notebooks 06–07.

## What you will be able to do
- Build a vocabulary from training text only.
- Pad variable-length sequences and mask padding.
- Understand the limits of averaging word embeddings.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 10](10_transfer_learning.ipynb) · [Next: 12](12_sequence_forecasting.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Prepare a deliberately small text task

Tokenization converts text into discrete symbols. An embedding learns a vector per token. This toy sentiment task teaches mechanics, not a deployable language model. The held-out sentences contain familiar sentiment words in new combinations; natural language has much more ambiguity.


In [ ]:
train_texts = ["good movie", "great story", "love this film", "excellent acting", "fun movie", "nice story",
               "bad movie", "awful story", "hate this film", "terrible acting", "boring movie", "poor story"]
train_labels = torch.tensor([1] * 6 + [0] * 6)
val_texts = ["good acting", "great film", "bad story", "boring film"]
val_labels = torch.tensor([1, 1, 0, 0])
vocab = {"<pad>": 0, "<unk>": 1}
for sentence in train_texts:
    for token in sentence.lower().split():
        if token not in vocab:
            vocab[token] = len(vocab)
def encode(sentence):
    return torch.tensor([vocab.get(word, 1) for word in sentence.lower().split()] or [1])
print(vocab)
assert encode("unseen").item() == vocab['<unk>']


## Step 3 — Collate variable-length examples

A DataLoader normally stacks equal shapes. A custom collate function pads each batch to its longest sequence. Padding is storage, not meaningful text, so exclude it from the average.


In [ ]:
from torch.nn.utils.rnn import pad_sequence
def collate(batch):
    sequences, labels = zip(*batch)
    return pad_sequence(sequences, batch_first=True, padding_value=0), torch.stack(labels)
dataset = [(encode(text), label) for text, label in zip(train_texts, train_labels)]
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate)
tokens, labels = next(iter(loader))
print("Token IDs:", tokens)

class TextClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), 12, padding_idx=0)
        self.head = nn.Linear(12, 2)
    def forward(self, tokens):
        mask = (tokens != 0).unsqueeze(-1)
        embedded = self.embedding(tokens)
        pooled = (embedded * mask).sum(1) / mask.sum(1).clamp_min(1)
        return self.head(pooled)

model = TextClassifier()
assert model(tokens).shape == (len(tokens), 2)


## Step 4 — Train, evaluate, and expose a limitation

Averaging ignores word order. It can work as a baseline but cannot represent all language relationships. On 12 training sentences, metrics are extremely noisy and vocabulary coverage is limited.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
for epoch in range(70):
    for tokens, labels in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(model(tokens), labels)
        loss.backward(); optimizer.step()
model.eval()
val_tokens = pad_sequence([encode(text) for text in val_texts], batch_first=True)
with torch.no_grad():
    probabilities = model(val_tokens).softmax(1)[:, 1]
for text, p in zip(val_texts, probabilities):
    print(f"{text:15s} -> P(positive)={p.item():.3f}")
print("Toy validation accuracy:", ((probabilities >= 0.5).long() == val_labels).float().mean().item())
a, b = encode("good movie"), encode("movie good")
with torch.no_grad():
    assert torch.allclose(model(a[None]), model(b[None]), atol=1e-6)


## Your turn — practice before opening the solutions

1. Check that adding padding does not change a prediction.
2. Count unknown tokens in 'wonderful new movie'.
3. Compare 'not good' with 'good'. Explain why this training set cannot teach reliable negation.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
example = encode("good movie")[None]
padded = nn.functional.pad(example, (0, 3), value=0)
with torch.no_grad():
    assert torch.allclose(model(example), model(padded), atol=1e-6)
    for text in ["not good", "good"]:
        print(text, model(encode(text)[None]).softmax(1).tolist())
assert (encode("wonderful new movie") == 1).sum().item() == 2
# 'not' is unknown; no training sentence supplies supervision for negation.


## Before moving on

A tokenizer, vocabulary, padding convention, and model belong together. Save all of them for inference. This model is order-invariant; the next lessons introduce sequence-aware models.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Word embeddings tutorial](https://docs.pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html)


[Course guide](../README.md) · [Previous: 10](10_transfer_learning.ipynb) · [Next: 12](12_sequence_forecasting.ipynb)
